# Time Series Forecasting Template

Use this notebook for forecasting when you have timestamped observations over time. It is especially suited for at least 2 to 3 years of data so seasonality can be evaluated more credibly.

Included approaches:
- Baseline lag model
- ARIMA
- SARIMA / SARIMAX
- Optional LSTM scaffold
- Optional Transformer scaffold


## When Each Model Makes Sense

- ARIMA: useful when the series is mostly driven by autocorrelation and trend
- SARIMA: useful when there is clear repeating seasonality
- LSTM: useful when you have enough history and nonlinear sequence effects
- Transformer: usually only worth it with larger sequence datasets and stronger modeling needs


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

try:
    from tensorflow import keras
    TENSORFLOW_AVAILABLE = True
except ImportError:
    TENSORFLOW_AVAILABLE = False

try:
    import torch
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False

plt.style.use('ggplot')


In [ ]:
DATA_PATH = Path('data.csv')
DATE_COL = 'signup_date'
TARGET_COL = 'order_value'
FREQ = 'D'
SEASONAL_PERIOD = 7

df = pd.read_csv(DATA_PATH)
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce')
ts_df = df[[DATE_COL, TARGET_COL]].dropna().sort_values(DATE_COL).copy()
ts_df = ts_df.set_index(DATE_COL).resample(FREQ).sum()
display(ts_df.head())


In [ ]:
ts_df.plot(figsize=(10, 4), title=f'{TARGET_COL} over time')
plt.ylabel(TARGET_COL)
plt.tight_layout()
plt.show()


In [ ]:
test_size = max(7, int(len(ts_df) * 0.2))
train = ts_df.iloc[:-test_size].copy()
test = ts_df.iloc[-test_size:].copy()

print({'train_rows': len(train), 'test_rows': len(test)})


In [ ]:
forecast_results = []

baseline_forecast = test[TARGET_COL].shift(1).fillna(train[TARGET_COL].iloc[-1])
forecast_results.append({
    'model': 'naive_lag_1',
    'mae': mean_absolute_error(test[TARGET_COL], baseline_forecast),
    'rmse': mean_squared_error(test[TARGET_COL], baseline_forecast, squared=False),
})

arima_model = ARIMA(train[TARGET_COL], order=(1, 1, 1)).fit()
arima_forecast = arima_model.forecast(steps=len(test))
forecast_results.append({
    'model': 'ARIMA(1,1,1)',
    'mae': mean_absolute_error(test[TARGET_COL], arima_forecast),
    'rmse': mean_squared_error(test[TARGET_COL], arima_forecast, squared=False),
})

sarima_model = SARIMAX(
    train[TARGET_COL],
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, SEASONAL_PERIOD),
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=False)
sarima_forecast = sarima_model.forecast(steps=len(test))
forecast_results.append({
    'model': f'SARIMA seasonal={SEASONAL_PERIOD}',
    'mae': mean_absolute_error(test[TARGET_COL], sarima_forecast),
    'rmse': mean_squared_error(test[TARGET_COL], sarima_forecast, squared=False),
})

forecast_results_df = pd.DataFrame(forecast_results).sort_values('rmse')
display(forecast_results_df)


In [ ]:
plot_df = pd.DataFrame({
    'actual': test[TARGET_COL],
    'baseline': baseline_forecast,
    'arima': arima_forecast,
    'sarima': sarima_forecast,
})

plot_df.plot(figsize=(10, 5), title='Forecast Comparison on Test Window')
plt.ylabel(TARGET_COL)
plt.tight_layout()
plt.show()


## Optional: LSTM Scaffold

Only use this when you have enough sequential history and want to test a nonlinear neural baseline. It requires more data and more tuning than ARIMA-family models.


In [ ]:
if TENSORFLOW_AVAILABLE:
    print('TensorFlow is available. Suggested next step: create rolling windows of length 14 to 60 and fit an LSTM.')
    print('Typical flow: scale series -> build sequences -> LSTM -> Dense(1) -> compare MAE/RMSE with ARIMA and SARIMA.')
else:
    print('Install tensorflow to run the LSTM section.')


## Optional: Transformer Scaffold

Transformers are usually overkill for small case studies, but they can be useful for richer multivariate forecasting or long sequence dependence. Use this only when the dataset size supports it and you need a stronger deep-learning benchmark.


In [ ]:
if TORCH_AVAILABLE:
    print('PyTorch is available. Suggested next step: build a dataset of fixed-length windows and compare a lightweight Transformer encoder against LSTM and SARIMA.')
else:
    print('Install torch to run the Transformer section.')


## Interpretation Checklist

Summarize:

- Which model performed best on the holdout period
- Whether the gains are material or just marginal
- Whether seasonality appears stable enough to trust
- What could break in production, such as regime change or promotions
- What business planning decision the forecast should support
